In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
orders_df = spark.read.parquet("data/external_table/part-00000-6109de51-a0e8-41af-839d-58e5ef769898-c000.snappy.parquet", inferSchema=True, header=True)

In [3]:
orders_df.createOrReplaceTempView('orders')

### 3. Number of orders for each status type

In [4]:
spark.sql("""
SELECT order_status, count(order_id) as order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC
""")

order_status,order_count
COMPLETE,22899
PENDING_PAYMENT,15030
PROCESSING,8275
PENDING,7610
CLOSED,7556
ON_HOLD,3798
SUSPECTED_FRAUD,1558
CANCELED,1428
PAYMENT_REVIEW,729


In [5]:
orders_df.groupBy('order_status').select('order_id').count()

AttributeError: 'GroupedData' object has no attribute 'select'

DataFrame → groupBy → GroupedData → agg/count/sum → DataFrame

Once .groupBy() is called, it is shifted from a "DataFrame" (a table of data) into a "GroupedData" object (a collection of buckets). A GroupedData object is "in limbo"—it doesn't have a .select() method because Spark is waiting to be told how to aggregate those buckets before it gives a table back.

``` 
from pyspark.sql.functions import count

orders_df.groupBy('order_status') \
         .agg(count('order_id').alias('order_count')) \
         .show() 
```

In [6]:
orders_df.groupBy('order_status').count()

order_status,count
PENDING_PAYMENT,15030
COMPLETE,22899
ON_HOLD,3798
PAYMENT_REVIEW,729
PROCESSING,8275
CLOSED,7556
SUSPECTED_FRAUD,1558
PENDING,7610
CANCELED,1428


In [7]:
orders_df.groupBy('order_status').count().sort('count', ascending=False)

order_status,count
COMPLETE,22899
PENDING_PAYMENT,15030
PROCESSING,8275
PENDING,7610
CLOSED,7556
ON_HOLD,3798
SUSPECTED_FRAUD,1558
CANCELED,1428
PAYMENT_REVIEW,729


In [ ]:
spark.stop()

### Both 'ascending=False' rule
```
orders_df.groupBy("customer_id") \
         .agg(count('order_id').alias("order_count")) \
         .sort(['order_count', 'customer_id'], ascending=False) \
         .show(10)
```


### One ascending and other descending
```
from pyspark.sql.functions import count, desc, asc
orders_df.groupBy("customer_id") \
         .agg(count('order_id').alias("order_count")) \
         .sort(desc("order_count"), asc("customer_id")) \
         .show(10)
```

### Equivalent SQL
```
spark.sql("""
        SELECT customer_id, COUNT(order_id) AS order_count
        FROM orders
        GROUP BY customer_id
        ORDER BY order_count DESC, customer_id ASC
        LIMIT 10
""")
```